In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath("/home/ElasticNotebook"))
%load_ext ElasticNotebook

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/new_notebooks/nyc-flight/rewritten/o4_mini_high_small/checkpoints/post_cell_15_try_1.pickle

In [ ]:
%%cudf.pandas.profile
### cell 16 ###

# filter to June
df = flights_df[flights_df['month'] == 6]

# use the original .agg with numpy mean to match the exact output
# (avoids subtle floating-point differences from the GPU .mean)
df = df.groupby('carrier', as_index=False).agg({
    'arr_delay': np.mean,
    'dep_delay': np.mean
})

# vectorized total_delay calculation
df['total_delay'] = df['arr_delay'] + df['dep_delay']

# find and print all carrier(s) with the minimum total_delay
min_total = df['total_delay'].min()
min_df = df[df['total_delay'] == min_total]
for c, t in zip(min_df['carrier'], min_df['total_delay']):
    print(c, t)

# return the DataFrame
df